# DP-SGD max_grad_norm sweep on UCI Adult

Notebook này bổ sung thí nghiệm nhỏ cho phase tiếp theo: giữ cố định `noise_multiplier = 1.5` và quét `max_grad_norm = [0.5, 1.0, 1.5, 2.0]`.

Mục tiêu không phải tăng accuracy bằng mọi giá, mà là quan sát clipping norm ảnh hưởng thế nào tới privacy-utility tradeoff của DP-SGD.


In [ ]:
# Run this cell if Opacus is not installed in the current notebook runtime.
# In Colab, torch is usually preinstalled; Opacus is the only extra library needed.
import importlib.util
import sys
import subprocess

if importlib.util.find_spec("opacus") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opacus"])


In [ ]:
from collections.abc import Sequence
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from opacus import PrivacyEngine
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("adult")
TRAIN_PATH = DATA_DIR / "adult.data"
TEST_PATH = DATA_DIR / "adult.test"

BATCH_SIZE = 256
EPOCHS = 20
LR_DP = 0.05
MOMENTUM = 0.9
DELTA = 1e-5
NOISE_MULTIPLIER = 1.5
MAX_GRAD_NORM_LIST = [0.5, 1.0, 1.5, 2.0]
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Device:", DEVICE)
print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)


## Load and preprocess data

Notebook dùng trực tiếp thư mục `adult/` trong repo. Test file của Adult có dòng metadata ở đầu và nhãn kết thúc bằng dấu chấm, nên loader xử lý cả hai chi tiết này.


In [ ]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income",
]

numeric_cols = [
    "age", "fnlwgt", "education_num", "capital_gain",
    "capital_loss", "hours_per_week",
]


def load_adult_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Put the UCI Adult files under an adult/ folder."
        )
    df = pd.read_csv(
        path,
        names=columns,
        skipinitialspace=True,
        na_values="?",
        comment="|",
    )
    df["income"] = df["income"].str.replace(".", "", regex=False)
    return df.dropna().reset_index(drop=True)


train_df = load_adult_csv(TRAIN_PATH)
test_df = load_adult_csv(TEST_PATH)

x_train_raw = train_df.drop(columns="income")
x_test_raw = test_df.drop(columns="income")
y_train = (train_df["income"] == ">50K").astype("float32")
y_test = (test_df["income"] == ">50K").astype("float32")

combined = pd.concat([x_train_raw, x_test_raw], axis=0)
categorical_cols = [col for col in combined.columns if col not in numeric_cols]
combined = pd.get_dummies(combined, columns=categorical_cols, dtype="float32")

x_train = combined.iloc[: len(train_df)].copy()
x_test = combined.iloc[len(train_df) :].copy()

means = x_train[numeric_cols].mean()
stds = x_train[numeric_cols].std().replace(0, 1)
x_train[numeric_cols] = (x_train[numeric_cols] - means) / stds
x_test[numeric_cols] = (x_test[numeric_cols] - means) / stds

x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values.reshape(-1, 1), dtype=torch.float32)
x_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1), dtype=torch.float32)

test_loader = DataLoader(
    TensorDataset(x_test_tensor, y_test_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Input features:", x_train_tensor.shape[1])


In [ ]:
class MLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: Sequence[int] = (128, 64, 32),
        output_dim: int = 1,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


criterion = nn.BCEWithLogitsLoss()


def evaluate(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    model.eval()
    correct = 0
    total = 0
    tp = 0
    fp = 0
    fn = 0

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(features)
            preds = (torch.sigmoid(logits) >= 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.numel()
            tp += ((preds == 1) & (labels == 1)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "accuracy": correct / total,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    }


def train_private(max_grad_norm: float) -> dict[str, float]:
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    model = MLP(input_dim=x_train_tensor.shape[1]).to(DEVICE)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LR_DP,
        momentum=MOMENTUM,
    )
    train_loader = DataLoader(
        TensorDataset(x_train_tensor, y_train_tensor),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    privacy_engine = PrivacyEngine(accountant="prv")
    model, optimizer, private_train_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=NOISE_MULTIPLIER,
        max_grad_norm=max_grad_norm,
    )

    start_time = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for features, labels in private_train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        metrics = evaluate(model, test_loader)
        epsilon = privacy_engine.get_epsilon(DELTA)
        print(
            f"norm={max_grad_norm:.1f} epoch={epoch:02d} "
            f"epsilon={epsilon:.4f} acc={metrics['accuracy']:.4f} "
            f"f1={metrics['f1_score']:.4f}"
        )

    elapsed = time.time() - start_time
    final_metrics = evaluate(model, test_loader)
    return {
        "max_grad_norm": max_grad_norm,
        "epsilon": privacy_engine.get_epsilon(DELTA),
        **final_metrics,
        "training_time": elapsed,
    }


In [ ]:
sweep_results = []

for max_grad_norm in MAX_GRAD_NORM_LIST:
    print("\n" + "=" * 72)
    print(f"Training DP-SGD with max_grad_norm={max_grad_norm}")
    print("=" * 72)
    sweep_results.append(train_private(max_grad_norm))

df_sweep = pd.DataFrame(sweep_results)
df_sweep.to_csv("uci_adult_dp_sgd_max_grad_norm_sweep.csv", index=False)
display(df_sweep)
print("Saved: uci_adult_dp_sgd_max_grad_norm_sweep.csv")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(df_sweep["max_grad_norm"], df_sweep["accuracy"], marker="o")
plt.xlabel("max_grad_norm")
plt.ylabel("Accuracy")
plt.title("Effect of Clipping Norm on Accuracy")
plt.grid(True)
plt.tight_layout()
plt.savefig("max_grad_norm_vs_accuracy.png", dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(df_sweep["max_grad_norm"], df_sweep["f1_score"], marker="o", color="tab:green")
plt.xlabel("max_grad_norm")
plt.ylabel("F1-score")
plt.title("Effect of Clipping Norm on F1-score")
plt.grid(True)
plt.tight_layout()
plt.savefig("max_grad_norm_vs_f1_score.png", dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(df_sweep["max_grad_norm"], df_sweep["training_time"], marker="o", color="tab:orange")
plt.xlabel("max_grad_norm")
plt.ylabel("Training time (seconds)")
plt.title("Effect of Clipping Norm on Training Time")
plt.grid(True)
plt.tight_layout()
plt.savefig("max_grad_norm_vs_training_time.png", dpi=200)
plt.show()


In [ ]:
def comment_for_norm(row: pd.Series) -> str:
    norm = row["max_grad_norm"]
    if norm == 0.5:
        return "Có thể clip quá mạnh, cần xem accuracy/F1 có giảm không."
    if norm == 1.0:
        return "Mốc hiện tại, dùng làm baseline của clipping norm."
    if norm == 1.5:
        return "Có thể giữ nhiều tín hiệu gradient hơn."
    return "Có thể tăng ảnh hưởng gradient trước khi thêm noise."


report_table = df_sweep.copy()
report_table["Nhận xét"] = report_table.apply(comment_for_norm, axis=1)
report_table = report_table.rename(
    columns={
        "max_grad_norm": "Max grad norm",
        "epsilon": "Epsilon",
        "accuracy": "Accuracy",
        "f1_score": "F1-score",
        "training_time": "Training time",
    }
)
display(report_table)
